## Download Landsat Image

In [1]:
# -*- coding: utf-8 -*-
"""
Bulk Landsat 8/9 Collection 2 Level-2 download for WRS Path/Row (Oahu).
-------------------------------------------------------------------------------
- Uses USGS M2M API (stable endpoint)
- If a manifest JSON already exists, skips search and uses it directly
- Otherwise searches (date range + bbox), then filters to WRS PATH/ROW using metadata
- Requests downloads, polls until ready, downloads .tar files in parallel
- Robust handling: no infinite wait when some downloads never become available
- Saves missing/failed entity IDs and can retry them automatically

Outputs:
  Y:/Mingyue/Oahu/landsat_c2_l2_tar/*.tar
  Y:/Mingyue/Oahu/landsat_c2_l2_tar/manifest_*.json
  Y:/Mingyue/Oahu/landsat_c2_l2_tar/missing_entityIds_*.json
  Y:/Mingyue/Oahu/landsat_c2_l2_tar/retry_report_*.json
"""

import os
import json
import time
import threading
from datetime import datetime
from pathlib import Path
from getpass import getpass

import requests
from tqdm.auto import tqdm


# =============================================================================
# USER SETTINGS
# =============================================================================
SITE = "West_Fl_Shelf"
OUT_DIR = Path("../") / SITE
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_DIR = OUT_DIR / "landsat_c2_l2_tar"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

YOUR_USGS_USERNAME = "mingyue752"
YOUR_M2M_TOKEN = os.environ.get("USGS_M2M_TOKEN")  # preferred; else prompt

START_DATE = "2020-01-01"
END_DATE   = "2026-01-01"

WRS_PATH = 17
WRS_ROW  = 42

# NOTE: This dataset name is what you used; keep if it works for you.
# If you ever need to switch, do it here.
DATASET_NAME = "landsat_ot_c2_l2"

# Oahu bbox (ONLY to reduce search size; does NOT crop downloads)
OAHU_MBR = {
    "filterType": "mbr",
    "lowerLeft": {"latitude": 25.171, "longitude": -83.475},
    "upperRight": {"latitude": 25.704, "longitude": -83.654},
}

# Manifest path (if file exists, search is skipped)
MANIFEST_PATH = DOWNLOAD_DIR / f"manifest_p{WRS_PATH:03d}r{WRS_ROW:03d}_{START_DATE}_to_{END_DATE}.json"


# =============================================================================
# DOWNLOAD / RETRY TUNING
# =============================================================================
MAX_THREADS = 5

# If M2M returns ready N/M but stalls, we break after STAGNANT_LIMIT polls
POLL_SECONDS = 30
STAGNANT_LIMIT = 6       # 6 polls * 30s = 3 minutes no progress => break & proceed
MAX_WAIT_SEC = 60 * 60   # hard cap per request label (1 hour)

# Retry missing entityIds automatically this many times
AUTO_RETRY_MISSING = True
MAX_RETRIES = 2          # additional request rounds for missing entityIds


# =============================================================================
# INTERNAL CONFIG
# =============================================================================
SERVICE_URL = "https://m2m.cr.usgs.gov/api/api/json/stable/"
REQUEST_TIMEOUT_SECONDS = 300

sema = threading.Semaphore(value=MAX_THREADS)


# =============================================================================
# M2M HELPERS
# =============================================================================
def send_request(endpoint: str, payload: dict | None, api_key: str | None = None):
    """POST to M2M API; return out['data'] or None on error."""
    url = SERVICE_URL + endpoint
    headers = {"X-Auth-Token": api_key} if api_key else {}
    data = json.dumps(payload) if payload is not None else None
    try:
        r = requests.post(url, data=data, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        r.raise_for_status()
        out = r.json()
        if out.get("errorCode"):
            raise RuntimeError(f"M2M API Error: {out['errorCode']} - {out.get('errorMessage')}")
        return out.get("data")
    except Exception as e:
        print(f"\n[ERROR] {endpoint}: {e}")
        return None


def download_file_usgs(url: str, display_id: str):
    """Download a tar file to DOWNLOAD_DIR/display_id.tar (resume-safe)."""
    sema.acquire()
    try:
        out_path = DOWNLOAD_DIR / f"{display_id}.tar"
        if out_path.exists() and out_path.stat().st_size > 1000:
            return  # already downloaded

        with requests.get(url, stream=True, timeout=3600) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))

            tmp_path = out_path.with_suffix(".tar.part")

            with open(tmp_path, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, unit_divisor=1024,
                desc=out_path.name, leave=False
            ) as bar:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

            tmp_path.replace(out_path)

        print(f"Downloaded: {out_path.name}")
    except Exception as e:
        print(f"\n[ERROR] download {display_id}: {e}")
    finally:
        sema.release()


# ---- metadata parsing (WRS info is inside 'metadata') ----
def _meta_get(scene: dict, field_name: str):
    want = field_name.strip().lower()
    for m in scene.get("metadata", []) or []:
        name = str(m.get("fieldName", "")).strip().lower()
        if name == want:
            return m.get("value")
    return None


def get_wrs_path_row(scene: dict):
    path = _meta_get(scene, "WRS Path") or _meta_get(scene, "Path")
    row  = _meta_get(scene, "WRS Row")  or _meta_get(scene, "Row")
    return path, row


# =============================================================================
# STEP 1: SEARCH SCENES (only if manifest not present) + FILTER TO PATH/ROW
# =============================================================================
def search_scenes_paged(api_key: str):
    print("\n--- Step 1: Searching USGS M2M scenes (paged) ---")
    all_results = []
    start_num = 1
    page_size = 1000

    scene_filter = {
        "acquisitionFilter": {"start": START_DATE, "end": END_DATE},
        "spatialFilter": OAHU_MBR,
    }

    total_hits = None

    while True:
        payload = {
            "datasetName": DATASET_NAME,
            "sceneFilter": scene_filter,
            "maxResults": page_size,
            "startingNumber": start_num,
            "sortField": "acquisitionDate",
            "sortDirection": "ASC",
        }

        data = send_request("scene-search", payload, api_key)
        if not data:
            print("\n[WARN] scene-search returned no data.")
            break

        results = data.get("results", [])
        if total_hits is None:
            total_hits = data.get("totalHits", 0)
            print(f"totalHits (after date+bbox filter): {total_hits}")
            if results:
                print("Example result keys:", list(results[0].keys()))

        all_results.extend(results)
        print(f"Fetched {len(all_results)} / {total_hits} ...", end="\r")

        if len(results) < page_size:
            break
        start_num += page_size

    print(f"\nTotal scenes returned by date+bbox filter: {len(all_results)}")

    filtered = []
    for r in all_results:
        p, q = get_wrs_path_row(r)
        if p is None or q is None:
            continue
        try:
            if int(p) == int(WRS_PATH) and int(q) == int(WRS_ROW):
                filtered.append(r)
        except Exception:
            continue

    print(f"Scenes after WRS filter P{WRS_PATH:03d}/R{WRS_ROW:03d}: {len(filtered)}")
    return filtered


def load_or_build_manifest(api_key: str):
    if MANIFEST_PATH.exists() and MANIFEST_PATH.stat().st_size > 100:
        print(f"\n[SKIP SEARCH] Using existing manifest:\n  {MANIFEST_PATH}")
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    scenes = search_scenes_paged(api_key)
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(scenes, f, indent=2)
    print(f"Saved manifest: {MANIFEST_PATH}")
    return scenes


# =============================================================================
# STEP 2: BUILD DOWNLOAD REQUEST LIST (entityId + productId)
# =============================================================================
def build_download_request_list(filtered_results: list[dict], api_key: str):
    """For each scene, pick first available download option and return downloads list + mapping."""
    downloads_to_request = []
    entity_to_display = {}  # entityId -> displayId (for naming)

    for r in tqdm(filtered_results, desc="Finding download products"):
        entity_id = r.get("entityId")
        display_id = r.get("displayId") or r.get("sceneId") or r.get("productId") or entity_id
        if not entity_id or not display_id:
            continue

        # If already downloaded, skip early
        out_path = DOWNLOAD_DIR / f"{display_id}.tar"
        if out_path.exists() and out_path.stat().st_size > 1000:
            continue

        opt_payload = {"datasetName": DATASET_NAME, "entityIds": [entity_id]}
        options = send_request("download-options", opt_payload, api_key)
        if not options:
            continue

        picked = None
        for opt in options:
            if opt.get("available") and opt.get("id"):
                picked = opt
                break
        if not picked:
            continue

        downloads_to_request.append({
            "entityId": picked["entityId"],
            "productId": picked["id"],
        })
        entity_to_display[str(picked["entityId"])] = str(display_id)

    return downloads_to_request, entity_to_display


def iter_ready_downloads(obj):
    """Yield (entityId, url) from possible ready lists in download-request/retrieve responses."""
    if not obj:
        return
    for key in ("availableDownloads", "available"):
        lst = obj.get(key, [])
        if isinstance(lst, list):
            for d in lst:
                if not isinstance(d, dict):
                    continue
                ent = d.get("entityId") or d.get("entity_id")
                url = d.get("url") or d.get("downloadUrl")
                if ent and url:
                    yield str(ent), url


def summarize_status_lists(status: dict):
    """Return counts for common status lists; names vary by API version."""
    def _get_list(*names):
        for n in names:
            v = status.get(n)
            if isinstance(v, list):
                return v
        return []

    preparing = _get_list("preparing", "preparingDownloads")
    failed    = _get_list("failed", "failedDownloads")
    removed   = _get_list("removed", "notAvailable", "unavailable")
    dup       = _get_list("duplicate", "duplicateDownloads")
    return preparing, failed, removed, dup


# =============================================================================
# STEP 3: REQUEST DOWNLOADS, POLL UNTIL READY OR STALLED, DOWNLOAD
# =============================================================================
def request_label(downloads_to_request: list[dict], api_key: str, label: str):
    req_payload = {"downloads": downloads_to_request, "label": label}
    return send_request("download-request", req_payload, api_key)


def poll_until_ready(label: str, downloads_to_request: list[dict], api_key: str):
    """Poll download-retrieve until ready OR stalled OR timeout. Returns dict: ent->url plus status snapshot."""
    retrieve_payload = {"label": label}
    download_urls = {}

    t0 = time.time()
    last_ready = -1
    stagnant_ticks = 0

    # first retrieve
    status = send_request("download-retrieve", retrieve_payload, api_key)
    if status:
        for ent, url in iter_ready_downloads(status):
            download_urls.setdefault(ent, url)

    while len(download_urls) < len(downloads_to_request):
        status = send_request("download-retrieve", retrieve_payload, api_key)
        if not status:
            time.sleep(20)
            continue

        for ent, url in iter_ready_downloads(status):
            download_urls.setdefault(ent, url)

        preparing, failed, removed, dup = summarize_status_lists(status)

        ready_now = len(download_urls)
        print(
            f"Ready: {ready_now}/{len(downloads_to_request)} | "
            f"Preparing: {len(preparing)} | Failed: {len(failed)} | Removed/NA: {len(removed)} | Dups: {len(dup)}",
            end="\r"
        )

        if ready_now == last_ready:
            stagnant_ticks += 1
        else:
            stagnant_ticks = 0
            last_ready = ready_now

        # Completed
        if ready_now == len(downloads_to_request):
            break

        # Key fix: if nothing preparing and no progress, break
        if len(preparing) == 0 and stagnant_ticks >= STAGNANT_LIMIT:
            print("\n[WARN] No progress and nothing preparing. Some downloads may be unavailable. Proceeding with what is ready.")
            break

        # Hard timeout
        if time.time() - t0 > MAX_WAIT_SEC:
            print("\n[WARN] Timed out waiting for all downloads. Proceeding with what is ready.")
            break

        time.sleep(POLL_SECONDS)

    print(f"\nDownloads ready for label={label}: {len(download_urls)}")
    return download_urls, status or {}


def download_urls_parallel(download_urls: dict, entity_to_display: dict):
    threads = []
    for ent, url in download_urls.items():
        display_id = entity_to_display.get(ent, ent)
        t = threading.Thread(target=download_file_usgs, args=(url, display_id))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()


def request_poll_download_round(downloads_to_request: list[dict], entity_to_display: dict, api_key: str, round_idx: int):
    if not downloads_to_request:
        print("\nNothing to request (everything already downloaded or no products available).")
        return set(), {"round": round_idx, "label": None, "requested": 0, "ready": 0, "missing": []}

    label = f"bulk_p{WRS_PATH:03d}r{WRS_ROW:03d}_{datetime.now().strftime('%Y%m%d%H%M%S')}_r{round_idx}"
    print(f"\n--- Step 2/3: Requesting downloads (round {round_idx}) ---")
    print(f"Will request {len(downloads_to_request)} downloads. Label={label}")

    req = request_label(downloads_to_request, api_key, label)
    if not req:
        print("[ERROR] download-request failed.")
        missing = {str(d["entityId"]) for d in downloads_to_request}
        return missing, {"round": round_idx, "label": label, "requested": len(downloads_to_request), "ready": 0, "missing": sorted(missing)}

    # initial ready from request response
    download_urls = dict(iter_ready_downloads(req))

    print("\n--- Step 3: Waiting for preparation ---")
    polled_urls, status_snapshot = poll_until_ready(label, downloads_to_request, api_key)
    download_urls.update(polled_urls)

    print("\n--- Step 4: Downloading .tar files ---")
    download_urls_parallel(download_urls, entity_to_display)

    requested_ents = {str(d["entityId"]) for d in downloads_to_request}
    missing = requested_ents - set(download_urls.keys())

    report = {
        "round": round_idx,
        "label": label,
        "requested": len(downloads_to_request),
        "ready": len(download_urls),
        "missing": sorted(missing),
        "download_dir": str(DOWNLOAD_DIR.resolve()),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "status_keys": list(status_snapshot.keys()) if isinstance(status_snapshot, dict) else [],
    }

    if missing:
        missing_path = DOWNLOAD_DIR / f"missing_entityIds_{label}.json"
        missing_path.write_text(json.dumps(sorted(missing), indent=2))
        print(f"[INFO] Missing entityIds saved to: {missing_path}")

    return missing, report


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    print("DOWNLOAD_DIR =", DOWNLOAD_DIR.resolve())
    print("MANIFEST_PATH =", MANIFEST_PATH.resolve())

    if not YOUR_M2M_TOKEN:
        YOUR_M2M_TOKEN = getpass("Enter your USGS M2M token: ")

    print("\n--- Logging in via token ---")
    api_key = send_request("login-token", {"username": YOUR_USGS_USERNAME, "token": YOUR_M2M_TOKEN})
    if not api_key:
        raise SystemExit("Login failed. Check username/token.")

    retry_reports = []
    try:
        scenes = load_or_build_manifest(api_key)

        print("\n--- Step 2: Preparing download requests (skip already-downloaded) ---")
        downloads_to_request, entity_to_display = build_download_request_list(scenes, api_key)

        # Round 0: request everything not downloaded
        missing, report = request_poll_download_round(downloads_to_request, entity_to_display, api_key, round_idx=0)
        retry_reports.append(report)

        # Retry missing entityIds if configured
        if AUTO_RETRY_MISSING and missing:
            for round_idx in range(1, MAX_RETRIES + 1):
                print(f"\n=== RETRY ROUND {round_idx} (missing={len(missing)}) ===")

                # Build a reduced downloads_to_request list for only missing entityIds
                # We need productId, so call download-options again for those entityIds.
                reduced = []
                for ent in tqdm(sorted(missing), desc=f"Rebuilding download list (retry {round_idx})"):
                    opt_payload = {"datasetName": DATASET_NAME, "entityIds": [ent]}
                    options = send_request("download-options", opt_payload, api_key)
                    if not options:
                        continue
                    picked = None
                    for opt in options:
                        if opt.get("available") and opt.get("id"):
                            picked = opt
                            break
                    if not picked:
                        continue
                    reduced.append({"entityId": picked["entityId"], "productId": picked["id"]})

                if not reduced:
                    print("[WARN] No retryable download products found for missing entityIds.")
                    break

                missing, report = request_poll_download_round(reduced, entity_to_display, api_key, round_idx=round_idx)
                retry_reports.append(report)

                if not missing:
                    print("\nAll missing downloads resolved.")
                    break

        # Save retry report summary
        report_path = DOWNLOAD_DIR / f"retry_report_p{WRS_PATH:03d}r{WRS_ROW:03d}_{START_DATE}_to_{END_DATE}.json"
        report_path.write_text(json.dumps(retry_reports, indent=2))
        print(f"\nSaved retry report: {report_path}")

        print("\nDone. Files are in:", DOWNLOAD_DIR.resolve())

    finally:
        send_request("logout", None, api_key)
        print("\nLogged out.")

DOWNLOAD_DIR = /data/purkislabmingyue/10.141.132.249/purkislab2a/Mingyue/West_Fl_Shelf/landsat_c2_l2_tar
MANIFEST_PATH = /data/purkislabmingyue/10.141.132.249/purkislab2a/Mingyue/West_Fl_Shelf/landsat_c2_l2_tar/manifest_p017r042_2020-01-01_to_2026-01-01.json


Enter your USGS M2M token:  ········



--- Logging in via token ---

--- Step 1: Searching USGS M2M scenes (paged) ---
totalHits (after date+bbox filter): 462
Example result keys: ['browse', 'cloudCover', 'entityId', 'displayId', 'orderingId', 'metadata', 'hasCustomizedMetadata', 'options', 'selected', 'spatialBounds', 'spatialCoverage', 'temporalCoverage', 'publishDate']
Fetched 462 / 462 ...
Total scenes returned by date+bbox filter: 462
Scenes after WRS filter P017/R042: 230
Saved manifest: ../West_Fl_Shelf/landsat_c2_l2_tar/manifest_p017r042_2020-01-01_to_2026-01-01.json

--- Step 2: Preparing download requests (skip already-downloaded) ---


Finding download products:   0%|          | 0/230 [00:00<?, ?it/s]


--- Step 2/3: Requesting downloads (round 0) ---
Will request 230 downloads. Label=bulk_p017r042_20260409192020_r0

--- Step 3: Waiting for preparation ---
Ready: 0/230 | Preparing: 0 | Failed: 0 | Removed/NA: 0 | Dups: 0
[WARN] No progress and nothing preparing. Some downloads may be unavailable. Proceeding with what is ready.

Downloads ready for label=bulk_p017r042_20260409192020_r0: 0

--- Step 4: Downloading .tar files ---


...

LC09_L2SP_017042_20251208_20251209_02_T2.tar:   0%|          | 0.00/811M [00:00<?, ?B/s]

LC09_L2SP_017042_20251224_20251226_02_T2.tar:   0%|          | 0.00/499M [00:00<?, ?B/s]

Downloaded: LC09_L2SP_017042_20251122_20251124_02_T2.tar
Downloaded: LC08_L2SP_017042_20251216_20251224_02_T2.tar
Downloaded: LC08_L2SP_017042_20251130_20251208_02_T2.tar
Downloaded: LC09_L2SP_017042_20251224_20251226_02_T2.tar


LC08_L2SP_017042_20260101_20260106_02_T2.tar:   0%|          | 0.00/820M [00:00<?, ?B/s]

Downloaded: LC09_L2SP_017042_20251208_20251209_02_T2.tar
Downloaded: LC08_L2SP_017042_20260101_20260106_02_T2.tar

Saved retry report: ../West_Fl_Shelf/landsat_c2_l2_tar/retry_report_p017r042_2020-01-01_to_2026-01-01.json

Done. Files are in: /data/purkislabmingyue/10.141.132.249/purkislab2a/Mingyue/West_Fl_Shelf/landsat_c2_l2_tar

Logged out.


## Extract data from .tar

In [2]:
from pathlib import Path
import os
import tarfile
from concurrent.futures import ThreadPoolExecutor, as_completed

SITE = "Timor_part2"
OUT_DIR = Path("../") / SITE

DOWNLOAD_DIR = OUT_DIR / "landsat_c2_l2_tar"

TAR_DIR = DOWNLOAD_DIR
OUT_DIR = Path("../") / SITE / "landsat_c2_l2_extracted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 4  # try 4 if Y:/ is a network drive
DELETE_TAR_AFTER_EXTRACT = False

def scene_already_extracted(scene_dir: Path) -> bool:
    return scene_dir.exists() and any(scene_dir.iterdir())

def extract_one(tar_path: Path):
    scene_name = tar_path.stem
    scene_dir = OUT_DIR / scene_name

    if scene_already_extracted(scene_dir):
        return tar_path.name, "SKIP"

    scene_dir.mkdir(parents=True, exist_ok=True)

    try:
        with tarfile.open(tar_path, "r") as tar:
            tar.extractall(scene_dir)

        if DELETE_TAR_AFTER_EXTRACT:
            tar_path.unlink(missing_ok=True)

        return tar_path.name, "OK"
    except tarfile.ReadError:
        return tar_path.name, "ERROR: ReadError"
    except Exception as e:
        return tar_path.name, f"ERROR: {e}"

tar_files = sorted(TAR_DIR.glob("*.tar"))
print("tar files:", len(tar_files))

ok = skip = err = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(extract_one, p) for p in tar_files]
    for fut in as_completed(futs):
        name, status = fut.result()
        if status == "OK":
            ok += 1
        elif status == "SKIP":
            skip += 1
        else:
            err += 1
        print(name, status)

print("OK", ok, "SKIP", skip, "ERR", err)
print("Output:", OUT_DIR)

tar files: 229


/tmp/ipykernel_133068/3499152056.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(scene_dir)


LC08_L2SP_109066_20200223_20200822_02_T1.tar OK
LC08_L2SP_109066_20200122_20200823_02_T1.tar OK
LC08_L2SP_109066_20200207_20200823_02_T1.tar OK
LC08_L2SP_109066_20200106_20200823_02_T1.tar OK
LC08_L2SP_109066_20200326_20200822_02_T2.tar OK
LC08_L2SP_109066_20200310_20200822_02_T1.tar OK
LC08_L2SP_109066_20200411_20200822_02_T1.tar OK
LC08_L2SP_109066_20200427_20200822_02_T1.tar OK
LC08_L2SP_109066_20200529_20200820_02_T1.tar OK
LC08_L2SP_109066_20200513_20200820_02_T1.tar OK
LC08_L2SP_109066_20200614_20200824_02_T1.tar OK
LC08_L2SP_109066_20200630_20200823_02_T1.tar OK
LC08_L2SP_109066_20200801_20200914_02_T1.tar OK
LC08_L2SP_109066_20200716_20200911_02_T1.tar OK
LC08_L2SP_109066_20200817_20200920_02_T1.tar OK
LC08_L2SP_109066_20200902_20200906_02_T1.tar OK
LC08_L2SP_109066_20201004_20201015_02_T1.tar OK
LC08_L2SP_109066_20200918_20201005_02_T1.tar OK
LC08_L2SP_109066_20201020_20201105_02_T1.tar OK
LC08_L2SP_109066_20201121_20210315_02_T1.tar OK
LC08_L2SP_109066_20201207_20210313_02_T2